# Importación de dataset

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn import metrics

In [2]:
df_portatiles_test = pd.read_csv("data/test.csv", sep=",")
df_portatiles_test.head()

,id,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,181,1098,HP,Spectre x360,Ultrabook,13.3,IPS Panel 4K Ultra HD 3840x2160,Intel Core i7 7500U 2.7GHz,16GB,512GB SSD,Intel HD Graphics 620,Windows 10,1.3kg
1,708,330,Acer,Aspire 5,Notebook,15.6,1366x768,AMD A12-Series 9720P 2.7GHz,8GB,256GB SSD,AMD Radeon RX 540,Windows 10,2.2kg
2,862,1260,Acer,Aspire ES1-572,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,500GB HDD,Intel HD Graphics 520,Linux,2.4kg
3,1064,1137,HP,EliteBook 1040,Notebook,14.0,Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 7,1.43kg
4,702,1015,HP,ENVY -,Notebook,13.3,IPS Panel Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.34kg


In [3]:
df_portatiles_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                391 non-null    int64  
 1   laptop_ID         391 non-null    int64  
 2   Company           391 non-null    object 
 3   Product           391 non-null    object 
 4   TypeName          391 non-null    object 
 5   Inches            391 non-null    float64
 6   ScreenResolution  391 non-null    object 
 7   Cpu               391 non-null    object 
 8   Ram               391 non-null    object 
 9   Memory            391 non-null    object 
 10  Gpu               391 non-null    object 
 11  OpSys             391 non-null    object 
 12  Weight            391 non-null    object 
dtypes: float64(1), int64(2), object(10)
memory usage: 39.8+ KB


# PROCESADO DE DATOS

| Columna          | ¿Qué hacer?                         |
|------------------|--------------------------------------|
| Company          | One-hot / Label                      |
| TypeName         | One-hot / Label                      |
| OpSys            | One-hot / Label                      |
| Gpu              | One-hot / Label                      |
| Product          | (opcional) One-hot                   |
| Ram              | Extraer número (GB)                  |
| Weight           | Extraer número (kg)                  |
| Memory           | Separar SSD/HDD y tamaños            |
| ScreenResolution | Extraer ancho/alto/IPS/Touch         |
| Cpu              | Extraer marca, modelo, GHz           |


In [4]:
# Procesado de columnas Ran y Weight
df_portatiles_test["Ram"] = df_portatiles_test["Ram"].str.replace("GB", "").astype(float)
df_portatiles_test["Weight"] = df_portatiles_test["Weight"].str.replace("kg", "").astype(float)

# Procesado de la columna ScreenResolution
resoluciones = df_portatiles_test['ScreenResolution'].str.extract(r'(\d+)x(\d+)')

df_portatiles_test['ScreenWidth'] = resoluciones[0].astype(int)
df_portatiles_test['ScreenHeight'] = resoluciones[1].astype(int)

df_portatiles_test["Resolucion"] = df_portatiles_test["ScreenWidth"] * df_portatiles_test["ScreenHeight"]

df_portatiles_test['Touchscreen'] = df_portatiles_test['ScreenResolution'].str.contains('Touchscreen', case=False, na=False).astype(int)
df_portatiles_test['IPS'] = df_portatiles_test['ScreenResolution'].str.contains('IPS', case=False, na=False).astype(int)

df_portatiles_test = df_portatiles_test.drop(columns=["ScreenResolution", "ScreenWidth", "ScreenHeight"], axis=1)

# Procesado de la columna Gpu
marca_cpu = df_portatiles_test["Cpu"].str.extract(r'^(Intel|AMD)')
df_portatiles_test["marca_cpu"] = marca_cpu

velocidad_cpu = df_portatiles_test["Cpu"].str.extract(r'(\d+\.?\d*)GHz')

df_portatiles_test["velocidad_cpu"] = velocidad_cpu.astype(float)

df_portatiles_test = df_portatiles_test.drop("Cpu", axis=1)

# Procesado de la columna Mempory
df_portatiles_test["total_almacenamiento_gb"] = (
    df_portatiles_test["Memory"]
        .str.extractall(r'(\d+)\s*(GB|TB)')
        .apply(lambda x: float(x[0]) * (1024 if x[1]=="TB" else 1), axis=1)
        .groupby(level=0).sum()
)

df_portatiles_test["ssd"] = df_portatiles_test['Memory'].str.contains(r'SSD', case=False).astype(int)
df_portatiles_test["hdd"] = df_portatiles_test['Memory'].str.contains(r'HDD', case=False).astype(int)
df_portatiles_test["flash"] = df_portatiles_test['Memory'].str.contains(r'Flash Storage', case=False).astype(int)

df_portatiles_test = df_portatiles_test.drop("Memory", axis=1)

# Procesado de la columna Gpu
df_portatiles_test["GPU_Intel"] = df_portatiles_test['Gpu'].str.contains(r'Intel', case=False).astype(int)
df_portatiles_test["GPU_AMD"] = df_portatiles_test['Gpu'].str.contains(r'AMD', case=False).astype(int)
df_portatiles_test["GPU_Nvidia"] = df_portatiles_test['Gpu'].str.contains(r'Nvidia', case=False).astype(int)

df_portatiles_test = df_portatiles_test.drop("Gpu", axis=1)

In [5]:
df_portatiles_test.head()

,id,laptop_ID,Company,Product,TypeName,Inches,Ram,OpSys,Weight,Resolucion,...,IPS,marca_cpu,velocidad_cpu,total_almacenamiento_gb,ssd,hdd,flash,GPU_Intel,GPU_AMD,GPU_Nvidia
0,181,1098,HP,Spectre x360,Ultrabook,13.3,16.0,Windows 10,1.30,8294400,...,1,Intel,2.7,512.0,1,0,0,1,0,0
1,708,330,Acer,Aspire 5,Notebook,15.6,8.0,Windows 10,2.20,1049088,...,0,AMD,2.7,256.0,1,0,0,0,1,0
2,862,1260,Acer,Aspire ES1-572,Notebook,15.6,4.0,Linux,2.40,1049088,...,0,Intel,2.0,500.0,0,1,0,1,0,0
3,1064,1137,HP,EliteBook 1040,Notebook,14.0,8.0,Windows 7,1.43,2073600,...,0,Intel,2.3,256.0,1,0,0,1,0,0
4,702,1015,HP,ENVY -,Notebook,13.3,8.0,Windows 10,1.34,2073600,...,1,Intel,2.5,256.0,1,0,0,1,0,0


In [6]:
df_portatiles_test.to_csv("data/test_kaggle_procesado.csv", index=False)